# TCGA-LUAD | STAR Gene Counts | Phases 1–4

**Live GDC download → gene-level parsing → sample metadata → QC and validation.**

Start with a small pilot; this notebook does not run alignment, differential expression or normalization.

In [ ]:
!pip -q install "requests>=2.31,<3" "pandas>=2.1,<3" "numpy>=1.26,<3" "matplotlib>=3.8,<4" "pytest>=8,<10"

## Upload the project ZIP
Upload `TCGA_LUAD_Project.zip` from your computer when prompted.

In [ ]:
from google.colab import files
import zipfile, os
uploaded = files.upload()
assert "TCGA_LUAD_Project.zip" in uploaded, "Upload the supplied TCGA_LUAD_Project.zip"
with zipfile.ZipFile("TCGA_LUAD_Project.zip") as z:
    z.extractall("/content")
os.chdir("/content/tcga_luad_project")
print("Project ready:", os.getcwd())

## Test parsers and validation logic (synthetic fixtures only)

In [ ]:
!python -m pytest -q

## Download and process pilot cohort
Default 8 primary tumors and 4 solid tissue normals. GDC files can be large; expect download time and storage usage. This requires internet access.

In [ ]:
from src.pipeline import run
TUMOR_N = 8
NORMAL_N = 4
summary = run("tcga_luad_output", tumor_n=TUMOR_N, normal_n=NORMAL_N)

## Inspect QC results

In [ ]:
import pandas as pd
from IPython.display import display, Image
qc = pd.read_csv("tcga_luad_output/qc/sample_qc.csv")
display(qc)
display(Image(filename="tcga_luad_output/qc/library_size.png"))
print(summary)

## Reproducibility snapshot (optional)
Run after success and save output alongside your results.

In [ ]:
!pip freeze > tcga_luad_output/provenance/pip_freeze.txt
print("Saved package versions to provenance/pip_freeze.txt")

## Package processed outputs and download
To keep the ZIP smaller, raw downloaded files are excluded. The manifest and checksums allow re-download.

In [ ]:
import shutil
from pathlib import Path
output = Path("tcga_luad_output")
with zipfile.ZipFile("/content/tcga_luad_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in output.rglob("*"):
        if p.is_file() and "raw" not in p.relative_to(output).parts:
            z.write(p, p.relative_to(output.parent))
files.download("/content/tcga_luad_results.zip")